In [1]:
import argparse
import copy

from datasets import load_dataset
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import GPT2Tokenizer
from tqdm.auto import tqdm
import wandb

from model import StoryNetwork

In [2]:
def train_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: str,
    epoch: int,
    use_wandb: bool = False
) -> float:
    """
    Trains model for one epoch
    
    Args:
        model: The neural network
        loader: DataLoader for training data
        optimizer: The optimizer
        device: Device to train on
        epoch: Current epoch number
        use_wandb: Whether to log to wandb
        
    Returns:
        Average loss for the epoch
    """
    model.train()
    total_loss = 0
    
    progress = tqdm(loader, desc=f'Training Epoch {epoch}')
    for idx, batch in enumerate(progress):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        target_ids = batch['labels'].to(device)
        
        optimizer.zero_grad()
        # Forward pass
        logits, _ = model(input_ids) # , attention_mask=attention_mask)
        
        # Calculate loss
        loss = nn.functional.cross_entropy(
            logits.view(-1, logits.size(-1)), 
            target_ids.view(-1),
            ignore_index=-100,
        )
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy ignoring padding tokens
        mask = (target_ids != -100)
        correct = (logits.argmax(dim=-1) == target_ids) * mask
        accuracy = correct.sum().float() / mask.sum()
    
        # Update progress bar
        progress.set_postfix({'loss': loss.item(), 'accuracy': accuracy.item()})
        
        # Log to wandb
        if use_wandb and idx % 100 == 0:
            wandb.log({
                'train_loss': loss.item(),
                'train_accuracy': accuracy.item(),
                'epoch': epoch
            })
            
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: str
) -> tuple[float, float]:
    """
    Evaluates model on validation set
    
    Args:
        model: The neural network
        loader: DataLoader for validation data
        device: Device to evaluate on
        
    Returns:
        Tuple of (average loss, average accuracy) on validation set
    """
    model.eval()
    total_loss = 0
    total_accuracy = 0
    
    for batch in tqdm(loader, desc='Evaluating'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        target_ids = batch['labels'].to(device)
        
        # Forward pass
        logits, _ = model(input_ids) # , attention_mask=attention_mask)
        
        # Calculate loss
        loss = nn.functional.cross_entropy(
            logits.view(-1, logits.size(-1)), 
            target_ids.view(-1),
            ignore_index=-100,
        )
        
        # Calculate accuracy ignoring padding tokens
        mask = (target_ids != -100)
        correct = (logits.argmax(dim=-1) == target_ids) * mask
        accuracy = correct.sum().float() / mask.sum()
        
        total_loss += loss.item()
        total_accuracy += accuracy.item()
        
    return total_loss / len(loader), total_accuracy / len(loader)


def collate_batch(batch, tokenizer, max_length):
    """
    Tokenizes and pads batch of text to longest sequence with ignored padding labels
    
    Args:
        batch: List of examples from dataset
        tokenizer: GPT2 tokenizer instance
        max_length: Maximum sequence length
        
    Returns:
        Dictionary with padded input_ids, labels and attention mask tensors
    """
    texts = [tokenizer.bos_token + example['text'] + tokenizer.eos_token for example in batch]
    
    # First tokenize without padding
    encoded = tokenizer(
        texts,
        truncation=True,
        max_length=max_length + 1,
        return_tensors=None  # Return list of token ids
    )

    input_ids = encoded['input_ids']
    labels = copy.deepcopy(input_ids)
    
    input_ids = [torch.tensor(x, dtype=torch.long) for x in input_ids]
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)

    labels = [torch.tensor(x, dtype=torch.long) for x in labels]
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)
    
    return {
        'input_ids': input_ids[:, :-1],
        'labels': labels[:, 1:],
        'attention_mask': input_ids.ne(tokenizer.pad_token_id)[:, :-1]
    }

In [3]:
parser = argparse.ArgumentParser()
parser.add_argument('--d_model', type=int, default=512)
parser.add_argument('--batch_size', type=int, default=32)
parser.add_argument('--learning_rate', type=float, default=3e-4)
parser.add_argument('--epochs', type=int, default=10)
parser.add_argument('--eval_every', type=int, default=1)
parser.add_argument('--use_wandb', action='store_true')
parser.add_argument('--max_length', type=int, default=128)

args = parser.parse_known_args([
    '--d_model', '512',
    '--batch_size', '32',
    '--learning_rate', '3e-4',
    '--epochs', '10',
    '--eval_every', '1',
    '--max_length', '128',
])[0]

In [4]:
# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
# Load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Initialize model
model = StoryNetwork(
    vocab_size=len(tokenizer),
    d_model=args.d_model
).to(device)

# Setup optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=args.learning_rate
)

In [5]:
# Initialize wandb if requested
if args.use_wandb:
    wandb.init(project='story-mingru')

# Load dataset
dataset = load_dataset('roneneldan/TinyStories')
dataset['train'] = dataset['train'].select(range(50000))

# Split into train and validation
val_size = min(1000, int(len(dataset['train']) * 0.1))
train_size = len(dataset['train']) - val_size
train_dataset, val_dataset = random_split(
    dataset['train'], 
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=args.batch_size,
    shuffle=True,
    num_workers=4,
    collate_fn=lambda b: collate_batch(b, tokenizer, args.max_length)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=args.batch_size,
    shuffle=False,
    num_workers=4,
    collate_fn=lambda b: collate_batch(b, tokenizer, args.max_length)
)

In [6]:
# Training loop
best_val_loss = float('inf')

for epoch in range(args.epochs):
    # Train
    train_loss = train_epoch(
        model, 
        train_loader, 
        optimizer, 
        device, 
        epoch,
        args.use_wandb
    )
    
    # Evaluate
    if epoch % args.eval_every == 0:
        val_loss, val_accuracy = evaluate(model, val_loader, device)
        print(f'\nEpoch {epoch}:')
        print(f'Train Loss: {train_loss:.4f}')
        print(f'Val Loss: {val_loss:.4f}')
        print(f'Val Accuracy: {val_accuracy:.4f}')
        
        if args.use_wandb:
            wandb.log({
                'val_loss': val_loss,
                'val_accuracy': val_accuracy,
                'epoch': epoch
            })
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            # torch.save(model.state_dict(), 'best_model.pt')

Training Epoch 0:   0%|          | 0/1532 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]


Epoch 0:
Train Loss: 3.5245
Val Loss: 2.8370
Val Accuracy: 0.4290


Training Epoch 1:   0%|          | 0/1532 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

...

Training Epoch 3:   0%|          | 0/1532 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]


Epoch 3:
Train Loss: 2.3836
Val Loss: 2.3805
Val Accuracy: 0.4825


Training Epoch 4:   0%|          | 0/1532 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f24795e3820>
Traceback (most recent call last):
  File "/home/edan/miniconda3/envs/intract/lib/python3.9/site-packages/torch/utils/data/dataloader.py", line 1478, in __del__
    self._shutdown_workers()
  File "/home/edan/miniconda3/envs/intract/lib/python3.9/site-packages/torch/utils/data/dataloader.py", line 1461, in _shutdown_workers
    if w.is_alive():
  File "/home/edan/miniconda3/envs/intract/lib/python3.9/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f24795e3820>
Traceback (most recent call last):
  File "/home/edan/miniconda3/envs/intract/lib/python3.9/site-packages/torch/utils/data/dataloader.py", line 1478, in __del__
    self._shutdown_workers()
  File "/home/edan/miniconda3/envs/intract/lib/python